# __MODEL_LABEL_MARKDOWN__

The visible cells own the data, features, transforms, validation choice,
model, review, and deployment decision. The imported helpers record model
identity, versions, dataset and split evidence, artifact hashes, lineage,
and rating-package rows.


In [ ]:
DATABASE_MODE = "local"  # "local" or "remote"
RUNTIME_MODULE = None  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False

DATA_AS_OF = None  # Or use MODEL.data_as_of_column below.
RUN_EDITOR = False
EDIT_REASON = ""
DEPLOY = False
DEPLOYMENT_REASON = ""


In [ ]:
from datetime import date
from pathlib import Path
import sys

search_root = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in (search_root, *search_root.parents)
        if (root / "pricing_pipeline").is_dir()
        and (root / "pricing_models").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the pricing repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from superglm import Numeric, SuperGLM  # noqa: E402

from pricing_pipeline.models.config import ValidationSplitConfig  # noqa: E402
from pricing_pipeline.notebook import (  # noqa: E402
    PricingModelSpec,
    build_candidate,
    connect,
    deploy_package,
    open_candidate,
    publish_candidate,
    publish_edits,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"


## Connect and verify the destination

Local mode creates persistent SQLite files under `.local`. Remote mode
obtains its private connection from the work runtime configured outside
this repository and refuses writes until the expected database matches.


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


## Load and transform the model frame

Replace the demo with the normal work query. Keep feature transforms as
ordinary visible Python and retain the primary key, target, exposure or
weights, optional split column, and data-as-of column in the final frame.


In [ ]:
rng = np.random.default_rng(42)
frame = pd.DataFrame({
    "__PRIMARY_KEY__": np.arange(1, 101),
    "__FEATURE_NAME__": rng.normal(size=100),
    "__EXPOSURE_NAME__": rng.uniform(0.25, 1.0, size=100),
    "data_as_of": [date.today()] * 100,
})
frame["__TARGET_NAME__"] = rng.poisson(
    frame["__EXPOSURE_NAME__"] * np.exp(-2.5 + 0.25 * frame["__FEATURE_NAME__"])
)
display({"Rows": len(frame), "Columns": len(frame.columns)})


## Define the model and validation decision


In [ ]:
MODEL = PricingModelSpec(
    name="__MODEL_NAME__",
    label="__MODEL_LABEL__",
    target="__TARGET_NAME__",
    model_type="__MODEL_TYPE__",
    deployment_slot="__DEPLOYMENT_SLOT__",
    features=("__FEATURE_NAME__",),
    dataset_name="__DATASET_NAME__",
    source_system="replace_with_source_name",
    pk_columns=("__PRIMARY_KEY__",),
    exposure_column="__EXPOSURE_NAME__",
    data_as_of_column="data_as_of",
    validation=ValidationSplitConfig.kfold(
        n_splits=5,
        random_state=42,
        shuffle=True,
    ),
)

def make_model():
    return SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        discrete=True,
        n_bins=64,
        features={"__FEATURE_NAME__": Numeric()},
    )

model = register_model(pricing, MODEL, source_root=MODEL_DIR)


## Fit and inspect the candidate


In [ ]:
candidate = build_candidate(
    pricing,
    model=model,
    frame=frame,
    model_factory=make_model,
    data_as_of=DATA_AS_OF,
)
candidate.metrics


## Publish the immutable candidate

Publication records the audit trail and creates a candidate package. It
does not change the live deployment.


In [ ]:
published = publish_candidate(pricing, candidate)
display({
    "Model": published.model_name,
    "Package": published.package_version,
    "State": published.package_status,
})


## Optional market edit and explicit review (remote mode only)


In [ ]:
reviewed = None
if RUN_EDITOR:
    reviewed = open_candidate(
        pricing,
        model=model,
        package_version=published.package_version,
    )
    display(reviewed.editor())
    if not EDIT_REASON.strip():
        raise ValueError("Describe the market or underwriting edit.")
    edited = publish_edits(
        pricing,
        candidate=reviewed,
        reason=EDIT_REASON,
    )
    reviewed = open_candidate(
        pricing,
        model=model,
        package_version=edited.package_version,
    )
    display({"Edited package": edited.package_version, "State": edited.package_status})


## Optional deployment of the reviewed package (remote mode only)


In [ ]:
if DEPLOY:
    if reviewed is None:
        reviewed = open_candidate(
            pricing,
            model=model,
            package_version=published.package_version,
        )
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the live package.")
    deployment = deploy_package(
        pricing,
        package=reviewed,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)
